#### SET UP

In [1]:
import pandas as pd
import sys
sys.path.append("..")

from src.data.ingestion import load_raw_data

df = load_raw_data()
print("Shape:", df.shape)

c:\Users\meenakshi\Documents\readmission-poc\venv\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Shape: (101766, 50)


In [2]:
from src.data.validation import validate_raw_data

validate_raw_data(df)
print("Validation passed.")

Validation passed.


In [3]:
from src.data.cleaning import clean_data

df_audited = clean_data(df)

print("Shape:", df_audited.shape)
print("Y distribution:\n", df_audited["Y"].value_counts(normalize=True))

Shape: (99343, 48)
Y distribution:
 Y
0    0.886112
1    0.113888
Name: proportion, dtype: float64


In [4]:
from src.data.splitting import split_by_patient

train_df, test_df = split_by_patient(df_audited)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train patients:", train_df["patient_nbr"].nunique())
print("Test patients:", test_df["patient_nbr"].nunique())

overlap = set(train_df["patient_nbr"]) & set(test_df["patient_nbr"])
print("Patient overlap between train/test:", len(overlap))

Train shape: (79541, 48)
Test shape: (19802, 48)
Train patients: 55992
Test patients: 13998
Patient overlap between train/test: 0


In [5]:
from src.data.splitting import split_by_patient

train_df, test_df = split_by_patient(df_audited)

PROCESSED_DIR = Path("../data/processed")
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print(f"Saved train.csv: {train_df.shape}")
print(f"Saved test.csv:  {test_df.shape}")

NameError: name 'Path' is not defined

In [ ]:
print(train_df["gender"].unique())
print(train_df["age"].unique())

<ArrowStringArray>
['Female', 'Male', 'Unknown/Invalid']
Length: 3, dtype: str
<ArrowStringArray>
[  '[0-10)',  '[10-20)',  '[20-30)',  '[30-40)',  '[40-50)',  '[50-60)',
  '[60-70)',  '[70-80)',  '[80-90)', '[90-100)']
Length: 10, dtype: str


In [ ]:
train_df["gender"].value_counts()

gender
Female             42809
Male               36729
Unknown/Invalid        3
Name: count, dtype: int64

In [ ]:
from src.features.demographics import build_demographic_features

demo_features = build_demographic_features(train_df)
print(demo_features.shape)
demo_features.head()

(79538, 9)


,race_AfricanAmerican,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_nan,gender_Female,gender_Male,age_midpoint
0,False,False,True,False,False,False,True,False,5
1,False,False,True,False,False,False,True,False,15
2,True,False,False,False,False,False,True,False,25
3,False,False,True,False,False,False,False,True,35
4,False,False,True,False,False,False,False,True,45


In [ ]:
train_df[["admission_type_id", "admission_source_id", "time_in_hospital",
          "num_lab_procedures", "num_procedures", "num_medications",
          "number_outpatient", "number_emergency", "number_inpatient",
          "number_diagnoses"]].describe()

,admission_type_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000,79541.000000
mean,2.032160,5.725374,4.378987,42.924027,1.330572,15.983656,0.369533,0.198677,0.631473,7.398562
std,1.446836,4.060024,2.965798,19.575774,1.700227,8.113520,1.244375,0.946064,1.248577,1.944777
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,3.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,8.000000,25.000000,14.000000,132.000000,6.000000,81.000000,40.000000,76.000000,21.000000,16.000000


In [ ]:
from src.features.encounter import build_encounter_features

encounter_features = build_encounter_features(train_df)
print(encounter_features.shape)
encounter_features.head()

(79541, 33)


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,admission_type_id_1,admission_type_id_2,...,admission_source_id_20,admission_source_id_22,admission_source_id_25,admission_source_id_3,admission_source_id_4,admission_source_id_5,admission_source_id_6,admission_source_id_7,admission_source_id_8,admission_source_id_9
0,1,41,0,1,0,0,0,1,False,False,...,False,False,False,False,False,False,False,False,False,False
1,3,59,0,18,0,0,0,9,True,False,...,False,False,False,False,False,False,False,True,False,False
2,2,11,5,13,2,0,1,6,True,False,...,False,False,False,False,False,False,False,True,False,False
3,2,44,1,16,0,0,0,7,True,False,...,False,False,False,False,False,False,False,True,False,False
4,1,51,0,8,0,0,0,5,True,False,...,False,False,False,False,False,False,False,True,False,False


In [7]:
from src.features.encounter import build_encounter_features

train_enc = build_encounter_features(train_df)
test_enc = build_encounter_features(test_df)

print("Train columns:", train_enc.shape[1])
print("Test columns:", test_enc.shape[1])

only_in_train = set(train_enc.columns) - set(test_enc.columns)
only_in_test = set(test_enc.columns) - set(train_enc.columns)
print("Columns only in train:", only_in_train)
print("Columns only in test:", only_in_test)

Train columns: 33
Test columns: 27
Columns only in train: {'admission_source_id_13', 'admission_source_id_25', 'admission_source_id_14', 'admission_type_id_4', 'admission_source_id_10', 'admission_source_id_11'}
Columns only in test: set()
